# Prediksi Produktivitas dan Produksi Padi – Provinsi Lampung
## Machine Learning Berbasis Agroklimatologi Fase Pertumbuhan dan Luas Panen (2019–2024)

**Tugas Mata Kuliah Big Data** | Teknik Informatika – Universitas Lampung – 2026

---

### Ringkasan Perbaikan v3

| Aspek | v2 (Sebelumnya) | v3 (Ini) |
|---|---|---|
| Kabupaten dalam model | **5 dari 15** (bug mapping nama) | **15 kabupaten** |
| Sampel efektif | **30** (seharusnya 90) | **90 sampel** |
| Metode evaluasi | LOYOCV (ada temporal leakage) | **Walk-Forward Validation** |
| Baseline model | Tidak ada | **Naive Mean + Naive Kab Mean** |
| Interpretasi R² produksi | Tidak dikontekstualisasikan | **Dijelaskan & dikontekstualisasikan** |

---

**Alur kerja:**
1. Data Loading dan Cleaning  
2. Rekayasa Fitur Agroklimatologi Berbasis Fase Pertumbuhan  
3. Exploratory Data Analysis (EDA)  
4. Pemodelan dengan Walk-Forward Validation  
5. Evaluasi Final dan Visualisasi


---
## Section 0 – Persiapan Lingkungan

In [ ]:
# Tidak ada dependency tambahan yang diinstal di notebook ini.
# Catatan refactor: `xgboost` sebelumnya di-install tetapi tidak digunakan oleh eksperimen,
# sehingga cell instalasi dihapus agar runtime lebih deterministik dan tidak bergantung jaringan.
print("Dependency tambahan tidak diinstal; eksperimen memakai pandas, scikit-learn, matplotlib, seaborn.")


In [ ]:
import io, json, copy, warnings, datetime, random, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path
from google.colab import files
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings('once')

# ── Gaya Visualisasi ─────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi'       : 130,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.grid'        : True,
    'grid.alpha'       : 0.3,
    'grid.linestyle'   : '--',
    'font.size'        : 10,
})
PALETTE_KAB = plt.cm.tab20.colors

# ── Reproducibility & Output Management ──────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

RESULT_DIR = Path('results') / 'Images'
RESULT_DIR.mkdir(parents=True, exist_ok=True)

PROCESSED_DIR_CUACA = Path('data') / 'Processed' / 'Cuaca'
PROCESSED_DIR_PADI  = Path('data') / 'Processed' / 'Padi'
PROCESSED_DIR_CUACA.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR_PADI.mkdir(parents=True, exist_ok=True)


def canonical_upload_stem(filename):
    """Normalisasi nama file upload Colab, termasuk suffix otomatis seperti ' (1)'."""
    return re.sub(r' \(\d+\)$', '', Path(filename).stem)


def validate_uploaded_files(uploaded, expected_count=None, suffix=None, required_stems=None, label='file'):
    """Validasi ringan untuk workflow Google Colab `files.upload()`.

    Fungsi ini sengaja tidak mengganti mekanisme upload manual. Tujuannya adalah
    mencegah eksperimen berjalan diam-diam dengan file kurang, format salah,
    atau nama JSON yang tidak cocok dengan mapping kabupaten.
    """
    if not uploaded:
        raise ValueError(f"Tidak ada {label} yang di-upload.")
    filenames = list(uploaded.keys())
    if expected_count is not None and len(filenames) != expected_count:
        raise ValueError(f"Jumlah {label} harus {expected_count}, tetapi diterima {len(filenames)}: {filenames}")
    if suffix is not None:
        invalid = [f for f in filenames if not f.lower().endswith(suffix.lower())]
        if invalid:
            raise ValueError(f"Ekstensi {label} tidak valid: {invalid}; expected '*{suffix}'.")
    if required_stems is not None:
        stems = {canonical_upload_stem(f) for f in filenames}
        missing = set(required_stems) - stems
        extra = stems - set(required_stems)
        if missing:
            raise ValueError(f"{label} wajib belum lengkap. Missing stems: {sorted(missing)}")
        if extra:
            raise ValueError(f"{label} tidak dikenal: {sorted(extra)}")
    return filenames


def assert_unique_key(df, key_cols, label):
    dup = int(df.duplicated(key_cols).sum())
    if dup:
        contoh = df.loc[df.duplicated(key_cols, keep=False), key_cols].head().to_dict('records')
        raise ValueError(f"{label}: ditemukan {dup} duplicate key pada {key_cols}. Contoh: {contoh}")


def save_figure(filename, **kwargs):
    """Simpan visualisasi secara konsisten ke results/Images."""
    path = RESULT_DIR / filename
    defaults = {'bbox_inches': 'tight', 'dpi': 130}
    defaults.update(kwargs)
    plt.savefig(path, **defaults)
    print(f"[SAVED] {path}")


# ── Konstanta Waktu ──────────────────────────────────────────────────────────
TAHUN_LIST  = list(range(2018, 2025))   # data BPS tersedia
TAHUN_FITUR = list(range(2019, 2025))   # tahun dengan fitur siklus tanam lengkap

# ── Parameter NASA POWER ─────────────────────────────────────────────────────
NASA_PARAMS = ['ALLSKY_SFC_SW_DWN', 'T2M', 'T2M_MAX', 'T2M_MIN', 'RH2M', 'PRECTOTCORR']
NAMA_PARAMS = {
    'ALLSKY_SFC_SW_DWN': 'Radiasi',
    'T2M'              : 'Suhu Avg',
    'T2M_MAX'          : 'Suhu Max',
    'T2M_MIN'          : 'Suhu Min',
    'RH2M'             : 'Kelembapan',
    'PRECTOTCORR'      : 'Curah Hujan',
}

# ── Kalender Musim Tanam Padi (Lampung) ──────────────────────────────────────
MUSIM = {
    'utama'  : {'label': 'Musim Tanam Utama (Panen Raya)',
                'keterangan': 'Nov T-1, Des T-1, Jan T, Feb T, Mar T',
                'warna': '#2ecc71'},
    'gadu'   : {'label': 'Musim Tanam Gadu (Panen Gadu)',
                'keterangan': 'Apr T, Mei T, Jun T, Jul T',
                'warna': '#f1c40f'},
    'kemarau': {'label': 'Musim Tanam Kemarau (Panen Kecil)',
                'keterangan': 'Agu T, Sep T, Okt T',
                'warna': '#e74c3c'},
}

# ── [FIX v3] Daftar 15 Kabupaten BPS (nama persis sesuai data BPS) ───────────
KABUPATEN_LIST = sorted([
    'Bandar Lampung', 'Lampung Barat',  'Lampung Selatan', 'Lampung Tengah',
    'Lampung Timur',  'Lampung Utara',  'Mesuji',          'Metro',
    'Pesawaran',      'Pesisir Barat',  'Pringsewu',       'Tanggamus',
    'Tulang Bawang',  'Tulang Bawang Barat', 'Way Kanan',
])

# ── [FIX v3] Mapping nama file JSON → nama kabupaten BPS ─────────────────────
# Solusi untuk bug: nama file JSON tidak berspasi (CamelCase), sedangkan
# nama kabupaten di BPS menggunakan spasi. Mapping eksplisit menghindari
# kesalahan rename otomatis yang hanya cocok untuk nama satu kata.
NAMA_FILE_KE_KAB = {
    'BandarLampung'    : 'Bandar Lampung',
    'LampungBarat'     : 'Lampung Barat',
    'LampungSelatan'   : 'Lampung Selatan',
    'LampungTengah'    : 'Lampung Tengah',
    'LampungTimur'     : 'Lampung Timur',
    'LampungUtara'     : 'Lampung Utara',
    'WayKanan'         : 'Way Kanan',
    'TulangBawang'     : 'Tulang Bawang',
    'TulangBawangBarat': 'Tulang Bawang Barat',
    'Pesawaran'        : 'Pesawaran',
    'Pringsewu'        : 'Pringsewu',
    'Mesuji'           : 'Mesuji',
    'Tanggamus'        : 'Tanggamus',
    'PesisirBarat'     : 'Pesisir Barat',
    'Metro'            : 'Metro',
}

print("Lingkungan siap.")
print(f"  Tahun data BPS    : {TAHUN_LIST[0]}–{TAHUN_LIST[-1]}")
print(f"  Tahun fitur model : {TAHUN_FITUR[0]}–{TAHUN_FITUR[-1]}")
print(f"  Kabupaten terdaftar: {len(KABUPATEN_LIST)}")
print(f"  Parameter cuaca   : {', '.join(NASA_PARAMS)}")


---
## Section 1 – Data Loading dan Cleaning

### 1.1 Data Produksi Padi dan Luas Panen (BPS)

Sumber: BPS Provinsi Lampung. Format CSV lebar dengan struktur:

| Kolom | Isi |
|---|---|
| Kolom 0 | Nama kabupaten/kota (15 wilayah) |
| Kolom 1–7 | Luas Panen (Ha) 2018–2024 |
| Kolom 8–14 | Produksi (Ton) 2018–2024 |

Dua baris pertama berisi header/sub-header dan dikecualikan secara otomatis.  
Baris "Provinsi Lampung" (total agregat) juga dikecualikan.

In [ ]:
# Upload file CSV BPS (PadiTahunan.csv)
# Workflow upload manual dipertahankan untuk kompatibilitas Google Colab.
uploaded_padi = files.upload()
padi_files = validate_uploaded_files(
    uploaded_padi,
    expected_count=1,
    suffix='.csv',
    label='file CSV BPS'
)
csv_bytes = uploaded_padi[padi_files[0]]
print(f"File diterima: {padi_files[0]} ({len(csv_bytes):,} bytes)")


In [ ]:
def parse_csv_padi(csv_bytes):
    """
    Membaca CSV BPS format lebar dengan multi-level header.

    Struktur kolom setelah pd.read_csv default:
      col[0]   : nama kabupaten/kota  ('Wilayah')
      col[1:8] : Luas Panen (Ha) 2018–2024
      col[8:15]: Produksi (Ton) 2018–2024
    """
    df_raw    = pd.read_csv(io.BytesIO(csv_bytes))
    col_kab   = df_raw.columns[0]
    data_cols = df_raw.columns[1:]
    if len(data_cols) != 14:
        raise ValueError(
            f"Format CSV BPS tidak sesuai: expected 14 kolom data (7 luas + 7 produksi), "
            f"tetapi ditemukan {len(data_cols)}."
        )
    luas_cols = data_cols[:7]
    prod_cols = data_cols[7:]

    # Filter: hanya baris kabupaten (exclude NaN dan baris total provinsi)
    mask = (df_raw[col_kab].notna() &
            ~df_raw[col_kab].astype(str).str.startswith('Provinsi'))
    df_kab = df_raw[mask].reset_index(drop=True)

    records = []
    for _, row in df_kab.iterrows():
        kab = str(row[col_kab]).strip()
        for j, tahun in enumerate(TAHUN_LIST):
            luas = pd.to_numeric(row[luas_cols[j]], errors='coerce')
            prod = pd.to_numeric(row[prod_cols[j]], errors='coerce')
            records.append({'kabupaten': kab, 'tahun': tahun,
                            'luas_panen_ha': luas, 'produksi_ton': prod})
    return pd.DataFrame(records)


def validate_padi_dataset(df):
    """Quality gate untuk dataset BPS setelah parsing."""
    expected_rows = len(KABUPATEN_LIST) * len(TAHUN_LIST)
    if df.shape[0] != expected_rows:
        raise ValueError(f"Dataset BPS harus {expected_rows} baris, tetapi ditemukan {df.shape[0]}.")
    missing_kab = set(KABUPATEN_LIST) - set(df['kabupaten'].unique())
    extra_kab = set(df['kabupaten'].unique()) - set(KABUPATEN_LIST)
    if missing_kab or extra_kab:
        raise ValueError(f"Mismatch kabupaten BPS. Missing={sorted(missing_kab)}, extra={sorted(extra_kab)}")
    assert_unique_key(df, ['kabupaten', 'tahun'], 'Dataset BPS')
    if df[['luas_panen_ha', 'produksi_ton']].isnull().any().any():
        raise ValueError("Dataset BPS mengandung nilai non-numeric/missing setelah parsing.")
    if (df['luas_panen_ha'] <= 0).any() or (df['produksi_ton'] <= 0).any():
        raise ValueError("Dataset BPS mengandung luas panen/produksi <= 0; target produktivitas tidak valid.")
    return True


df_produksi_luas = parse_csv_padi(csv_bytes)
validate_padi_dataset(df_produksi_luas)

print(f"Dataset BPS : {df_produksi_luas.shape[0]} baris x {df_produksi_luas.shape[1]} kolom")
print(f"Kabupaten   : {df_produksi_luas['kabupaten'].nunique()} wilayah")
print(f"Tahun       : {df_produksi_luas['tahun'].min()}–{df_produksi_luas['tahun'].max()}")
print(f"Nilai hilang: {df_produksi_luas.isnull().sum().sum()}")
print("[OK] Quality gate BPS: schema, key uniqueness, kabupaten, dan nilai positif valid.")
print()
print(df_produksi_luas.groupby('tahun')[['luas_panen_ha','produksi_ton']]
      .sum().round(0).to_string())


### 1.2 Data Cuaca NASA POWER

Sumber: NASA POWER API (parameter harian, 2018–2024).  
Setiap kabupaten diwakili oleh satu koordinat representatif (centroid).

> **[FIX v3]** Nama kabupaten kini dipetakan menggunakan `NAMA_FILE_KE_KAB` —
> kamus eksplisit yang mencocokkan nama file CamelCase (mis. `LampungTengah.json`)
> dengan nama resmi BPS (`Lampung Tengah`). Perbaikan ini memastikan semua
> 15 kabupaten berhasil di-*merge*, bukan hanya 5 kabupaten nama satu kata.

In [ ]:
# Upload semua 15 file JSON cuaca sekaligus
# Workflow upload manual dipertahankan untuk kompatibilitas Google Colab.
uploaded_cuaca = files.upload()
cuaca_files = validate_uploaded_files(
    uploaded_cuaca,
    expected_count=15,
    suffix='.json',
    required_stems=NAMA_FILE_KE_KAB.keys(),
    label='file JSON cuaca NASA POWER'
)
print(f"{len(cuaca_files)} file diterima dan lolos validasi nama:")
for fname in sorted(cuaca_files):
    print(f"  {fname} ({len(uploaded_cuaca[fname]):,} bytes)")


In [ ]:
def parse_json_nasa(uploaded_dict):
    """
    Membaca file JSON NASA POWER hasil upload manual.

    Quality gate yang diterapkan:
    - nama file harus cocok dengan NAMA_FILE_KE_KAB,
    - seluruh NASA_PARAMS harus tersedia,
    - periode harus 2018-01-01 s.d. 2024-12-31,
    - duplicate key kabupaten-tanggal tidak diperbolehkan.
    """
    records = []
    expected_start = pd.Timestamp(f'{TAHUN_LIST[0]}-01-01')
    expected_end   = pd.Timestamp(f'{TAHUN_LIST[-1]}-12-31')

    for fname, raw_bytes in uploaded_dict.items():
        fname_key = canonical_upload_stem(fname).strip()
        kab = NAMA_FILE_KE_KAB.get(fname_key)
        if kab is None:
            raise ValueError(f"File cuaca tidak dikenal: {fname_key}. Periksa NAMA_FILE_KE_KAB.")

        try:
            payload = json.loads(raw_bytes)
        except json.JSONDecodeError as exc:
            raise ValueError(f"Tidak dapat membaca JSON: {fname}") from exc

        data_params = payload.get('properties', {}).get('parameter', {})
        missing_params = set(NASA_PARAMS) - set(data_params.keys())
        if missing_params:
            raise ValueError(f"{fname}: parameter NASA POWER hilang: {sorted(missing_params)}")

        tanggal_set = set(data_params[NASA_PARAMS[0]].keys())
        for p in NASA_PARAMS[1:]:
            if set(data_params[p].keys()) != tanggal_set:
                raise ValueError(f"{fname}: tanggal parameter {p} tidak konsisten dengan {NASA_PARAMS[0]}.")

        tanggal_list = sorted(tanggal_set)
        if pd.to_datetime(tanggal_list[0], format='%Y%m%d') != expected_start:
            raise ValueError(f"{fname}: tanggal awal tidak sesuai expected {expected_start.date()}.")
        if pd.to_datetime(tanggal_list[-1], format='%Y%m%d') != expected_end:
            raise ValueError(f"{fname}: tanggal akhir tidak sesuai expected {expected_end.date()}.")

        for tgl in tanggal_list:
            tgl_dt = pd.to_datetime(tgl, format='%Y%m%d')
            row = {
                'kabupaten': kab,
                'tanggal'  : tgl_dt,
                'tahun'    : tgl_dt.year,
                'bulan'    : tgl_dt.month,
                'hari'     : tgl_dt.day,
            }
            for p in NASA_PARAMS:
                val = data_params[p].get(tgl, np.nan)
                row[p] = np.nan if val in (-999, -99, None) else float(val)
            records.append(row)

    df = (pd.DataFrame(records)
          .sort_values(['kabupaten', 'tanggal'])
          .reset_index(drop=True))
    assert_unique_key(df, ['kabupaten', 'tanggal'], 'Dataset cuaca harian')
    return df


df_harian = parse_json_nasa(uploaded_cuaca)

print()
print(f"Data harian : {df_harian.shape[0]:,} baris")
print(f"Kabupaten   : {df_harian['kabupaten'].nunique()} "
      f"({sorted(df_harian['kabupaten'].unique())})")
print(f"Periode     : {df_harian['tanggal'].min().date()} s.d. "
      f"{df_harian['tanggal'].max().date()}")
print()
print("Nilai hilang per parameter:")
print(df_harian[NASA_PARAMS].isnull().sum().to_string())

expected_daily_rows = len(KABUPATEN_LIST) * len(pd.date_range(f'{TAHUN_LIST[0]}-01-01', f'{TAHUN_LIST[-1]}-12-31'))
if df_harian.shape[0] != expected_daily_rows:
    raise ValueError(f"Jumlah baris harian tidak sesuai: expected {expected_daily_rows}, got {df_harian.shape[0]}.")

loaded = set(df_harian['kabupaten'].unique())
missing = set(KABUPATEN_LIST) - loaded
if missing:
    raise ValueError(f"Kabupaten belum ter-load: {sorted(missing)}")
print()
print("[OK] Semua 15 kabupaten berhasil dimuat dan lolos quality gate harian.")


### 1.3 Penanganan Nilai Hilang

Nilai `-999` (sentinel NASA POWER) sudah dikonversi ke `NaN` pada tahap parsing.  
Imputasi menggunakan median bulanan per kabupaten — lebih robust terhadap outlier
dibanding mean, dan sesuai untuk distribusi cuaca yang bisa condong (skewed).

In [ ]:
def imputasi_median_bulanan(df):
    """Imputasi NaN dengan median historis kabupaten-bulan yang time-aware.

    Catatan metodologis:
    - Data NASA POWER saat ini tidak memiliki missing value, sehingga fungsi ini
      biasanya dilewati.
    - Jika missing muncul pada update data, nilai diisi dengan median observasi
      terdahulu untuk kabupaten-bulan yang sama. Fallback median kabupaten-bulan
      hanya dipakai bila tidak ada histori sebelumnya dan akan dilaporkan.
    """
    df = df.copy().sort_values(['kabupaten', 'tanggal']).reset_index(drop=True)
    total_nan_awal = int(df[NASA_PARAMS].isnull().sum().sum())
    if total_nan_awal == 0:
        print("Tidak ada nilai hilang — imputasi dilewati.")
        return df

    fallback_count = 0
    for p in NASA_PARAMS:
        miss_before = int(df[p].isnull().sum())
        if miss_before == 0:
            continue
        for (_, _), idx in df.groupby(['kabupaten', 'bulan']).groups.items():
            idx = pd.Index(idx)
            series = df.loc[idx, p]
            historical_median = series.expanding(min_periods=1).median().shift(1)
            fallback = series.median()
            fill_values = historical_median.fillna(fallback)
            mask_missing = df.loc[idx, p].isnull()
            fallback_count += int((mask_missing & historical_median.isnull()).sum())
            df.loc[idx[mask_missing], p] = fill_values.loc[idx[mask_missing]]
        miss_after = int(df[p].isnull().sum())
        print(f"  {p}: missing {miss_before} → {miss_after}")

    sisa = int(df[NASA_PARAMS].isnull().sum().sum())
    print(f"Nilai hilang total: {total_nan_awal} → {sisa} setelah imputasi time-aware.")
    if fallback_count:
        print(f"[CATATAN] {fallback_count} nilai memakai fallback median kabupaten-bulan karena tidak ada histori sebelumnya.")
    if sisa:
        raise ValueError("Masih ada missing value setelah imputasi; hentikan pipeline.")
    return df


df_harian = imputasi_median_bulanan(df_harian)


### 1.4 Agregasi Bulanan untuk Audit dan Pembanding Musiman Lama

Data harian tetap menjadi sumber utama feature engineering agroklimatologi. Agregasi bulanan di bawah ini dipertahankan sebagai artefak audit dan sebagai basis pembanding metode lama pada ablation study. Dengan demikian, revisi tidak menghapus analisis sebelumnya, tetapi memisahkan jelas antara:

- **fitur musiman lama**: agregasi kasar per musim tanam;
- **fitur agroklimatologi baru**: indikator harian-mingguan berbasis fase pertumbuhan padi.


In [ ]:
# Curah hujan: SUM (ketersediaan air kumulatif)
# Parameter lain: MEAN (kondisi rata-rata)
AGG_RULES = {p: 'sum' if p == 'PRECTOTCORR' else 'mean' for p in NASA_PARAMS}

df_bulanan = (
    df_harian
    .groupby(['kabupaten', 'tahun', 'bulan'])[NASA_PARAMS]
    .agg(AGG_RULES)
    .reset_index()
)

n_kab      = df_bulanan['kabupaten'].nunique()
n_expected = n_kab * 12 * len(TAHUN_LIST)
print(f"Data bulanan : {df_bulanan.shape[0]} baris  "
      f"({n_kab} kab × 12 bulan × {len(TAHUN_LIST)} tahun)")
print(f"Baris diharapkan: {n_expected} | Selisih: {n_expected - df_bulanan.shape[0]}")


### 1.5 Simpan Dataset Intermediate

In [ ]:
cuaca_out = PROCESSED_DIR_CUACA / 'cuaca_bulanan_clean.csv'
padi_out  = PROCESSED_DIR_PADI / 'produksi_luas_clean.csv'
df_bulanan.to_csv(cuaca_out, index=False)
df_produksi_luas.to_csv(padi_out, index=False)
print("Dataset intermediate disimpan:")
print(f"  {cuaca_out}")
print(f"  {padi_out}")


---
## Section 2 – Rekayasa Fitur Agroklimatologi Berbasis Fase Pertumbuhan

Pendekatan lama yang hanya merata-ratakan cuaca selama musim tanam berisiko menghilangkan sinyal biologis penting. Pada revisi ini data NASA POWER tetap digunakan pada resolusi **harian**, lalu diubah menjadi urutan 16 minggu yang merepresentasikan satu siklus tanam padi.

Asumsi operasional yang digunakan:

- satu siklus tanam sekitar **16 minggu**;
- kalender representatif dimulai **1 November tahun sebelumnya**, sesuai awal musim hujan dan musim tanam utama di Lampung;
- minggu 1–16 dipetakan menjadi fase: vegetatif awal, vegetatif akhir, generatif, pengisian bulir, dan pematangan;
- fitur yang dimodelkan dibatasi pada indikator paling bermakna agar tidak terjadi dimensionality explosion pada dataset sekitar 90 sampel.

Secara biologis, dampak cuaca berbeda antar fase. Stres panas pada fase generatif dapat meningkatkan risiko gabah hampa, kekeringan beruntun pada fase vegetatif dapat membatasi anakan, kelembapan tinggi beruntun dapat menaikkan risiko penyakit, dan radiasi rendah saat pengisian bulir dapat menurunkan akumulasi biomassa ke bulir.


In [ ]:
# ── Parameter agroklimatologi ────────────────────────────────────────────────
PLANTING_MONTH_DAY = (11, 1)       # siklus representatif mulai 1 Nov T-1
N_WEEKS_CYCLE      = 16
DAYS_PER_WEEK      = 7
DRY_DAY_MM         = 1.0
HEAVY_RAIN_3D_MM   = 50.0
HEAT_STRESS_C      = 35.0
COLD_NIGHT_C       = 20.0          # ambang konservatif malam dingin padi tropis
GDD_BASE_C         = 10.0
RH_HIGH_PCT        = 85.0
RH_DISEASE_DAYS    = 7
LOW_RAD_MJ         = 12.0          # MJ/m²/hari; indikator radiasi rendah berkepanjangan
EXTREME_RAD_Q      = 0.20          # ambang relatif untuk hari radiasi sangat rendah

FASE_MINGGU = {
    'vegetatif_awal' : list(range(1, 4)),
    'vegetatif_akhir': list(range(4, 7)),
    'generatif'      : list(range(7, 11)),
    'pengisian_bulir': list(range(11, 15)),
    'pematangan'     : list(range(15, 17)),
}

FASE_LABEL = {
    'vegetatif_awal' : 'Vegetatif awal (minggu 1–3)',
    'vegetatif_akhir': 'Vegetatif akhir/anakan (minggu 4–6)',
    'generatif'      : 'Generatif (minggu 7–10)',
    'pengisian_bulir': 'Pengisian bulir (minggu 11–14)',
    'pematangan'     : 'Pematangan (minggu 15–16)',
}


def longest_true_run(values):
    """Panjang maksimum deret True berturut-turut."""
    max_run = run = 0
    for val in values:
        if bool(val):
            run += 1
            max_run = max(max_run, run)
        else:
            run = 0
    return int(max_run)


def count_rolling_events(series, window=3, threshold=50.0):
    """Jumlah jendela rolling dengan akumulasi melewati threshold.

    Ini menghitung kejadian hidrologis pendek, bukan total hari. Rolling window
    memakai min_periods=window agar hanya jendela 3 hari penuh yang dihitung.
    """
    roll = series.rolling(window=window, min_periods=window).sum()
    return int((roll > threshold).sum())


def bangun_fitur_musiman_lama(df_bulanan, nasa_params):
    """Fitur musiman lama dipertahankan sebagai pembanding ablation.

    Jendela:
      Utama   : Nov T-1 s.d. Mar T
      Gadu    : Apr T s.d. Jul T
      Kemarau : Agu T s.d. Okt T
    """
    windows = {
        'utama'  : lambda t: [(t-1, 11), (t-1, 12), (t, 1), (t, 2), (t, 3)],
        'gadu'   : lambda t: [(t, m) for m in [4, 5, 6, 7]],
        'kemarau': lambda t: [(t, m) for m in [8, 9, 10]],
    }

    def agg_window(df_kab, window, suffix):
        rows = []
        for (y, m) in window:
            r = df_kab[(df_kab['tahun'] == y) & (df_kab['bulan'] == m)]
            if r.empty:
                return None
            rows.append(r[nasa_params].values[0])
        arr = np.array(rows)
        fitur = {}
        for j, p in enumerate(nasa_params):
            agg = arr[:, j].sum() if p == 'PRECTOTCORR' else arr[:, j].mean()
            label = 'sum' if p == 'PRECTOTCORR' else 'mean'
            fitur[f'old_{p}_{suffix}_{label}'] = agg
        return fitur

    records = []
    for kab in sorted(df_bulanan['kabupaten'].unique()):
        df_kab = df_bulanan[df_bulanan['kabupaten'] == kab]
        for tahun in TAHUN_FITUR:
            rec = {'kabupaten': kab, 'tahun': tahun}
            valid = True
            for musim_name, window_fn in windows.items():
                f = agg_window(df_kab, window_fn(tahun), musim_name)
                if f is None:
                    valid = False
                    break
                rec.update(f)
            if valid:
                records.append(rec)
    return pd.DataFrame(records)


def weekly_agro_features(df_week, week_no, low_rad_threshold=LOW_RAD_MJ):
    """Ekstraksi indikator agroklimatologi satu minggu pertumbuhan."""
    rain = df_week['PRECTOTCORR'].astype(float)
    tmax = df_week['T2M_MAX'].astype(float)
    tmin = df_week['T2M_MIN'].astype(float)
    tavg = df_week['T2M'].astype(float)
    rh   = df_week['RH2M'].astype(float)
    rad  = df_week['ALLSKY_SFC_SW_DWN'].astype(float)

    dry_mask = rain < DRY_DAY_MM
    rh_high_mask = rh > RH_HIGH_PCT

    return {
        f'week_{week_no:02d}_rain_total'       : rain.sum(),
        f'week_{week_no:02d}_rain_days'        : int((rain >= DRY_DAY_MM).sum()),
        f'week_{week_no:02d}_dry_spell_max'    : longest_true_run(dry_mask),
        f'week_{week_no:02d}_rain_extreme3d'   : count_rolling_events(rain, 3, HEAVY_RAIN_3D_MM),
        f'week_{week_no:02d}_rain_std'         : rain.std(ddof=0),
        f'week_{week_no:02d}_tmean_mean'       : tavg.mean(),
        f'week_{week_no:02d}_tmax_max'         : tmax.max(),
        f'week_{week_no:02d}_heat_days'        : int((tmax > HEAT_STRESS_C).sum()),
        f'week_{week_no:02d}_cold_nights'      : int((tmin < COLD_NIGHT_C).sum()),
        f'week_{week_no:02d}_gdd_sum'          : np.maximum(tavg - GDD_BASE_C, 0).sum(),
        f'week_{week_no:02d}_tmean_std'        : tavg.std(ddof=0),
        f'week_{week_no:02d}_rh_mean'          : rh.mean(),
        f'week_{week_no:02d}_rh_high_days'     : int(rh_high_mask.sum()),
        f'week_{week_no:02d}_rh_high_spell_max': longest_true_run(rh_high_mask),
        f'week_{week_no:02d}_disease_risk'     : int(longest_true_run(rh_high_mask) > RH_DISEASE_DAYS),
        f'week_{week_no:02d}_rad_mean'         : rad.mean(),
        f'week_{week_no:02d}_rad_low_days'     : int((rad < low_rad_threshold).sum()),
        f'week_{week_no:02d}_rad_roll3_min'    : rad.rolling(3, min_periods=1).mean().min(),
    }


def phase_agro_features(df_cycle):
    """Agregasi indikator terarah per fase pertumbuhan padi."""
    fitur = {}
    for fase, weeks in FASE_MINGGU.items():
        dff = df_cycle[df_cycle['week_no'].isin(weeks)]
        rain = dff['PRECTOTCORR'].astype(float)
        tmax = dff['T2M_MAX'].astype(float)
        tmin = dff['T2M_MIN'].astype(float)
        tavg = dff['T2M'].astype(float)
        rh   = dff['RH2M'].astype(float)
        rad  = dff['ALLSKY_SFC_SW_DWN'].astype(float)

        fitur.update({
            f'phase_{fase}_rain_total'       : rain.sum(),
            f'phase_{fase}_dry_spell_max'    : longest_true_run(rain < DRY_DAY_MM),
            f'phase_{fase}_rain_extreme3d'   : count_rolling_events(rain, 3, HEAVY_RAIN_3D_MM),
            f'phase_{fase}_heat_days'        : int((tmax > HEAT_STRESS_C).sum()),
            f'phase_{fase}_cold_nights'      : int((tmin < COLD_NIGHT_C).sum()),
            f'phase_{fase}_gdd_sum'          : np.maximum(tavg - GDD_BASE_C, 0).sum(),
            f'phase_{fase}_rh_high_days'     : int((rh > RH_HIGH_PCT).sum()),
            f'phase_{fase}_rh_high_spell_max': longest_true_run(rh > RH_HIGH_PCT),
            f'phase_{fase}_disease_risk'     : int(longest_true_run(rh > RH_HIGH_PCT) > RH_DISEASE_DAYS),
            f'phase_{fase}_rad_mean'         : rad.mean(),
            f'phase_{fase}_rad_low_days'     : int((rad < LOW_RAD_MJ).sum()),
            f'phase_{fase}_rad_roll7_mean'   : rad.rolling(7, min_periods=1).mean().mean(),
        })

    # Fitur interaksi biologis yang sengaja sedikit dan interpretatif.
    fitur['stress_generatif_heat_days'] = fitur['phase_generatif_heat_days']
    fitur['stress_generatif_hot_and_dry_days'] = int(
        ((df_cycle['week_no'].isin(FASE_MINGGU['generatif'])) &
         (df_cycle['T2M_MAX'] > HEAT_STRESS_C) &
         (df_cycle['PRECTOTCORR'] < DRY_DAY_MM)).sum()
    )
    fitur['stress_pengisian_rad_low_days'] = fitur['phase_pengisian_bulir_rad_low_days']
    fitur['stress_pematangan_rain_extreme3d'] = fitur['phase_pematangan_rain_extreme3d']
    fitur['stress_cycle_disease_risk_any'] = int(
        max(fitur[f'phase_{fase}_disease_risk'] for fase in FASE_MINGGU) > 0
    )
    return fitur


def bangun_fitur_agroklimatologi(df_harian):
    """Bangun fitur sequence-aware 16 minggu dan agregasi fase pertumbuhan."""
    # Ambang relatif radiasi sangat rendah dihitung dari seluruh data harian agar stabil.
    rad_extreme_threshold = df_harian['ALLSKY_SFC_SW_DWN'].quantile(EXTREME_RAD_Q)

    records = []
    for kab in sorted(df_harian['kabupaten'].unique()):
        df_kab = df_harian[df_harian['kabupaten'] == kab].sort_values('tanggal')
        for tahun in TAHUN_FITUR:
            start = pd.Timestamp(year=tahun-1, month=PLANTING_MONTH_DAY[0], day=PLANTING_MONTH_DAY[1])
            end   = start + pd.Timedelta(days=N_WEEKS_CYCLE * DAYS_PER_WEEK - 1)
            cycle = df_kab[(df_kab['tanggal'] >= start) & (df_kab['tanggal'] <= end)].copy()
            if len(cycle) != N_WEEKS_CYCLE * DAYS_PER_WEEK:
                continue

            cycle['day_in_cycle'] = (cycle['tanggal'] - start).dt.days + 1
            cycle['week_no'] = ((cycle['day_in_cycle'] - 1) // DAYS_PER_WEEK + 1).astype(int)

            rec = {
                'kabupaten': kab,
                'tahun': tahun,
                'tanggal_tanam_asumsi': start,
                'tanggal_panen_asumsi': end,
                'rad_extreme_threshold': rad_extreme_threshold,
            }

            for week_no in range(1, N_WEEKS_CYCLE + 1):
                df_week = cycle[cycle['week_no'] == week_no]
                rec.update(weekly_agro_features(df_week, week_no, LOW_RAD_MJ))
                rec[f'week_{week_no:02d}_rad_extreme_low_days'] = int(
                    (df_week['ALLSKY_SFC_SW_DWN'] < rad_extreme_threshold).sum()
                )

            rec.update(phase_agro_features(cycle))
            records.append(rec)

    return pd.DataFrame(records)


# Fitur baru (agroklimatologi harian-mingguan) dan fitur lama (musiman) dibangun berdampingan.
df_cuaca_agro = bangun_fitur_agroklimatologi(df_harian)
df_cuaca_musiman_lama = bangun_fitur_musiman_lama(df_bulanan, NASA_PARAMS)

KOLOM_CUACA_AGRO_ALL = [c for c in df_cuaca_agro.columns
                        if c not in ['kabupaten', 'tahun', 'tanggal_tanam_asumsi',
                                     'tanggal_panen_asumsi', 'rad_extreme_threshold']]
KOLOM_CUACA_LAMA = [c for c in df_cuaca_musiman_lama.columns if c not in ['kabupaten', 'tahun']]

# Subset model dibatasi untuk menjaga rasio sampel/fitur.
WEEKLY_MODEL_METRICS = [
    'rain_total', 'dry_spell_max', 'heat_days', 'rh_high_days', 'rad_mean'
]
KOLOM_WEEKLY_MODEL = [
    f'week_{w:02d}_{metric}'
    for w in range(1, N_WEEKS_CYCLE + 1)
    for metric in WEEKLY_MODEL_METRICS
]
KOLOM_PHASE_MODEL = [
    c for c in KOLOM_CUACA_AGRO_ALL
    if c.startswith('phase_') and (
        c.endswith('_rain_total') or c.endswith('_dry_spell_max') or
        c.endswith('_rain_extreme3d') or c.endswith('_heat_days') or
        c.endswith('_rh_high_days') or c.endswith('_disease_risk') or
        c.endswith('_rad_mean') or c.endswith('_rad_low_days')
    )
]
KOLOM_STRESS_MODEL = [c for c in KOLOM_CUACA_AGRO_ALL if c.startswith('stress_')]
KOLOM_CUACA_AGRO_MODEL = KOLOM_WEEKLY_MODEL + KOLOM_PHASE_MODEL + KOLOM_STRESS_MODEL
KOLOM_CUACA = KOLOM_CUACA_AGRO_MODEL

print(f"Fitur agroklimatologi harian-mingguan: {df_cuaca_agro.shape[0]} baris × {len(KOLOM_CUACA_AGRO_ALL)} fitur kandidat")
print(f"Fitur agro terpilih untuk model       : {len(KOLOM_CUACA_AGRO_MODEL)} fitur")
print(f"Fitur musiman lama untuk pembanding    : {len(KOLOM_CUACA_LAMA)} fitur")
print(f"Tahun tersedia                         : {sorted(df_cuaca_agro['tahun'].unique())}")
print()
print("Fase pertumbuhan yang digunakan:")
for fase, label in FASE_LABEL.items():
    print(f"  {label}")
print()
print("Contoh fitur sequence-aware:")
for c in KOLOM_WEEKLY_MODEL[:15]:
    print(f"  {c}")


### 2.2 Penggabungan dengan Luas Panen, Fitur Historis, dan Target

Fitur cuaca baru digabungkan dengan produksi dan luas panen BPS pada level kabupaten-tahun. Target tetap **produktivitas ton/ha**, karena target ini lebih tepat untuk menilai efek agroklimatologi dibanding produksi total yang sangat dipengaruhi luas panen.

Fitur historis produktivitas tetap dipertahankan secara time-aware. Ini penting karena produktivitas padi dipengaruhi faktor lokasi yang persisten seperti tanah, irigasi, varietas, dan praktik budidaya yang tidak seluruhnya terukur oleh NASA POWER.


In [ ]:
# Gabungkan fitur cuaca agroklimatologi baru, fitur musiman lama, produksi, dan luas panen.
df_fitur = (
    df_cuaca_agro
    .merge(df_cuaca_musiman_lama, on=['kabupaten', 'tahun'], how='inner', validate='one_to_one')
    .merge(
        df_produksi_luas[['kabupaten', 'tahun', 'luas_panen_ha', 'produksi_ton']],
        on=['kabupaten', 'tahun'],
        how='inner',
        validate='one_to_one'
    )
)

# Target utama: produktivitas (ton/ha) — menghilangkan efek skala luas lahan.
if (df_fitur['luas_panen_ha'] <= 0).any():
    raise ValueError("Terdapat luas_panen_ha <= 0; target produktivitas tidak dapat dihitung.")
df_fitur['produktivitas_ton_per_ha'] = df_fitur['produksi_ton'] / df_fitur['luas_panen_ha']

# Fitur historis target yang time-aware (hanya memakai data sebelum tahun prediksi).
df_hist_prodvt = df_produksi_luas.copy()
df_hist_prodvt['produktivitas_ton_per_ha_hist'] = (
    df_hist_prodvt['produksi_ton'] / df_hist_prodvt['luas_panen_ha']
)


def get_hist_prodvt_features(kabupaten, tahun):
    hist = (df_hist_prodvt[(df_hist_prodvt['kabupaten'] == kabupaten) &
                           (df_hist_prodvt['tahun'] < tahun)]
            .sort_values('tahun')['produktivitas_ton_per_ha_hist'])
    if hist.empty:
        raise ValueError(f"Tidak ada histori produktivitas untuk {kabupaten} sebelum {tahun}.")
    return pd.Series({
        'prodvt_lag1' : hist.iloc[-1],
        'prodvt_roll2': hist.tail(2).mean(),
        'prodvt_roll3': hist.tail(3).mean(),
    })

hist_features = df_fitur.apply(
    lambda r: get_hist_prodvt_features(r['kabupaten'], r['tahun']),
    axis=1
)
df_fitur = pd.concat([df_fitur, hist_features], axis=1)
HIST_FEATURES = ['prodvt_lag1', 'prodvt_roll2', 'prodvt_roll3']

n_sampel = df_fitur.shape[0]
n_kab    = df_fitur['kabupaten'].nunique()
expected_samples = len(KABUPATEN_LIST) * len(TAHUN_FITUR)

print(f"Matriks fitur final: {n_sampel} sampel × {df_fitur.shape[1]} kolom")
print(f"  Kabupaten              : {n_kab}  (diharapkan 15)")
print(f"  Tahun                  : {sorted(df_fitur['tahun'].unique())}")
print(f"  Fitur agro kandidat    : {len(KOLOM_CUACA_AGRO_ALL)}")
print(f"  Fitur agro untuk model : {len(KOLOM_CUACA_AGRO_MODEL)}")
print(f"  Fitur musiman lama     : {len(KOLOM_CUACA_LAMA)}")
print(f"  Fitur histori          : {len(HIST_FEATURES)} (lag/rolling produktivitas)")
print(f"  Target                 : produktivitas_ton_per_ha")
print()

if n_sampel != expected_samples:
    missing_keys = (
        pd.MultiIndex.from_product([KABUPATEN_LIST, TAHUN_FITUR], names=['kabupaten', 'tahun'])
        .difference(pd.MultiIndex.from_frame(df_fitur[['kabupaten', 'tahun']]))
    )
    raise ValueError(f"Sampel hasil merge tidak lengkap. Missing keys: {list(missing_keys)[:10]}")
assert_unique_key(df_fitur, ['kabupaten', 'tahun'], 'Matriks fitur final')
required_numeric = (KOLOM_CUACA_AGRO_MODEL + KOLOM_CUACA_LAMA + HIST_FEATURES +
                    ['luas_panen_ha', 'produksi_ton', 'produktivitas_ton_per_ha'])
if df_fitur[required_numeric].isnull().any().any():
    raise ValueError("Matriks fitur final mengandung missing value.")
if not np.isfinite(df_fitur['produktivitas_ton_per_ha']).all():
    raise ValueError("Target produktivitas mengandung nilai non-finite.")
print("[OK] Semua 15 kabupaten berhasil di-merge dan quality gate fitur final valid.")

print()
print("Statistik target (produktivitas ton/ha):")
desc = df_fitur['produktivitas_ton_per_ha'].describe()
print(desc.round(3).to_string())
cv = desc['std'] / desc['mean'] * 100
print(f"  Koefisien Variasi (CV): {cv:.1f}%  ← MAPE baseline naif kira-kira mengikuti variasi ini")


### 2.3 Persiapan Matriks Model

One-Hot Encoding kabupaten tetap digunakan untuk menangkap efek lokasi. Namun fitur cuaca default kini memakai subset agroklimatologi terpilih, bukan seluruh kandidat mingguan. Pembatasan ini sengaja dilakukan karena jumlah sampel hanya sekitar 90; fitur yang terlalu banyak dapat membuat model terlihat kompleks tetapi tidak stabil pada walk-forward validation.


In [ ]:
# One-Hot Encoding kabupaten.
MODEL_BASE_COLUMNS = (KOLOM_CUACA_AGRO_MODEL + KOLOM_CUACA_LAMA + HIST_FEATURES +
                      ['luas_panen_ha', 'kabupaten', 'tahun',
                       'produktivitas_ton_per_ha', 'produksi_ton'])
df_model = pd.get_dummies(
    df_fitur[MODEL_BASE_COLUMNS],
    columns=['kabupaten'],
    drop_first=True
)

KOLOM_KAB_OHE = [c for c in df_model.columns if c.startswith('kabupaten_')]
FITUR_HIST    = ['prodvt_lag1', 'prodvt_roll2']
FITUR_HIST_EXTENDED = HIST_FEATURES

FITUR_AGRO_ONLY      = KOLOM_CUACA_AGRO_MODEL
FITUR_OLD_WEATHER    = KOLOM_CUACA_LAMA
FITUR_AGRO_HIST      = FITUR_HIST + KOLOM_CUACA_AGRO_MODEL
FITUR_OLD_HIST       = FITUR_HIST + KOLOM_CUACA_LAMA
FITUR_HIST_KAB       = FITUR_HIST + KOLOM_KAB_OHE
FITUR_AGRO_HIST_KAB  = FITUR_HIST + KOLOM_CUACA_AGRO_MODEL + KOLOM_KAB_OHE
FITUR_OLD_HIST_KAB   = FITUR_HIST + KOLOM_CUACA_LAMA + KOLOM_KAB_OHE
FITUR_MODEL          = FITUR_AGRO_HIST_KAB
FITUR_AGRO_FULL      = FITUR_AGRO_HIST_KAB + ['luas_panen_ha']
FITUR_OLD_FULL       = FITUR_OLD_HIST_KAB + ['luas_panen_ha']

X      = df_model[FITUR_MODEL].values.astype(float)
y      = df_model['produktivitas_ton_per_ha'].values
GROUPS = df_fitur['tahun'].values   # untuk Walk-Forward CV

print(f"Dimensi X default : {X.shape}  (sampel × fitur)")
print(f"Dimensi y         : {y.shape}")
print(f"Jumlah fitur default: {len(FITUR_MODEL)}")
print(f"  Cuaca agro terpilih : {len(KOLOM_CUACA_AGRO_MODEL)}")
print(f"  Cuaca musiman lama  : {len(KOLOM_CUACA_LAMA)} tersedia untuk ablation")
print(f"  Histori target      : {len(HIST_FEATURES)} tersedia | {len(FITUR_HIST)} dipakai model ringkas")
print(f"  Kab OHE             : {len(KOLOM_KAB_OHE)}  (15 kab - 1 referensi)")
print(f"  Luas panen          : hanya dipakai pada eksperimen full tertentu")
print()
print(f"Rasio sampel/fitur default: {X.shape[0]}/{len(FITUR_MODEL)} = "
      f"{X.shape[0]/len(FITUR_MODEL):.2f}:1")
print("  Catatan: rasio ini rendah; karena itu evaluasi tetap walk-forward, baseline tetap dipakai, dan ablation wajib dibaca bersama metrik.")


---
## Section 3 – Exploratory Data Analysis (EDA)

### 3.1 Statistik Deskriptif per Kabupaten

In [ ]:
stat_kab = df_fitur.groupby('kabupaten').agg(
    produksi_mean  =('produksi_ton',            'mean'),
    produksi_std   =('produksi_ton',            'std'),
    luas_mean      =('luas_panen_ha',           'mean'),
    prodvt_mean    =('produktivitas_ton_per_ha', 'mean'),
    prodvt_std     =('produktivitas_ton_per_ha', 'std'),
).round(2).sort_values('produksi_mean', ascending=False)

print("Statistik Produksi, Luas Panen, dan Produktivitas per Kabupaten (2019–2024)")
print("=" * 85)
print(stat_kab.rename(columns={
    'produksi_mean': 'Produksi Rata2 (ton)',
    'produksi_std' : 'Produksi StdDev',
    'luas_mean'    : 'Luas Rata2 (ha)',
    'prodvt_mean'  : 'Produktivitas Rata2 (t/ha)',
    'prodvt_std'   : 'Produktivitas StdDev',
}).to_string())


### 3.2 Tren Produktivitas per Kabupaten (2019–2024)

In [ ]:
kabupaten_list = sorted(df_fitur['kabupaten'].unique())
n_kab = len(kabupaten_list)
ncols, nrows = 3, 5

fig, axes = plt.subplots(nrows, ncols, figsize=(15, 18))
fig.suptitle('Tren Produktivitas Padi per Kabupaten/Kota\nProvinsi Lampung 2019–2024',
             fontsize=14, fontweight='bold', y=1.01)

for idx, kab in enumerate(kabupaten_list):
    ax    = axes[idx // ncols][idx % ncols]
    df_k  = df_fitur[df_fitur['kabupaten'] == kab].sort_values('tahun')
    color = PALETTE_KAB[idx % len(PALETTE_KAB)]
    ax.plot(df_k['tahun'], df_k['produktivitas_ton_per_ha'],
            marker='o', linewidth=2, markersize=5, color=color)
    ax.fill_between(df_k['tahun'], df_k['produktivitas_ton_per_ha'],
                    alpha=0.12, color=color)
    ax.set_title(kab, fontsize=9, fontweight='bold')
    ax.set_xlabel('Tahun', fontsize=8)
    ax.set_ylabel('ton/ha', fontsize=8)
    ax.set_xticks(TAHUN_FITUR)
    ax.set_xticklabels([str(t)[2:] for t in TAHUN_FITUR], fontsize=7)
    ax.tick_params(axis='y', labelsize=7)

for i in range(n_kab, nrows * ncols):
    axes[i // ncols][i % ncols].set_visible(False)

plt.tight_layout()
save_figure('tren_produktivitas.png')
plt.show()


### 3.3 Distribusi Produktivitas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram seluruh sampel
ax   = axes[0]
vals = df_fitur['produktivitas_ton_per_ha']
ax.hist(vals, bins=20, color='steelblue', edgecolor='white', linewidth=0.7)
ax.axvline(vals.mean(), color='crimson', linestyle='--', linewidth=1.5,
           label=f'Rata-rata: {vals.mean():.2f} ton/ha')
ax.axvline(vals.mean() - vals.std(), color='darkorange', linestyle=':', linewidth=1.2,
           label=f'±1σ: [{vals.mean()-vals.std():.2f}, {vals.mean()+vals.std():.2f}]')
ax.axvline(vals.mean() + vals.std(), color='darkorange', linestyle=':', linewidth=1.2)
ax.set_title('Distribusi Produktivitas\n(Seluruh Kabupaten, 2019–2024)', fontsize=11)
ax.set_xlabel('Produktivitas (ton/ha)')
ax.set_ylabel('Frekuensi')
ax.legend(fontsize=9)

# Boxplot per kabupaten
ax           = axes[1]
median_order = (df_fitur.groupby('kabupaten')['produktivitas_ton_per_ha']
                .median().sort_values(ascending=False).index)
df_box = [df_fitur[df_fitur['kabupaten'] == k]['produktivitas_ton_per_ha'].values
          for k in median_order]
bp = ax.boxplot(df_box, vert=False, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6),
                medianprops=dict(color='navy', linewidth=2))
ax.set_yticks(range(1, len(median_order) + 1))
ax.set_yticklabels(median_order, fontsize=8)
ax.set_title('Distribusi Produktivitas per Kabupaten (2019–2024)', fontsize=11)
ax.set_xlabel('Produktivitas (ton/ha)')

plt.tight_layout()
save_figure('distribusi_produktivitas.png')
plt.show()


### 3.4 Profil Agroklimatologi Mingguan Siklus Tanam

Visualisasi berikut mengganti fokus dari rata-rata musiman menjadi dinamika mingguan. Tujuannya bukan menambah kompleksitas, tetapi melihat kapan stres air, panas, kelembapan, dan radiasi muncul dalam siklus pertumbuhan.


In [ ]:
weekly_summary = []
for w in range(1, N_WEEKS_CYCLE + 1):
    row = {
        'week_no': w,
        'rain_total_mean' : df_fitur[f'week_{w:02d}_rain_total'].mean(),
        'dry_spell_mean'  : df_fitur[f'week_{w:02d}_dry_spell_max'].mean(),
        'heat_days_mean'  : df_fitur[f'week_{w:02d}_heat_days'].mean(),
        'rh_high_mean'    : df_fitur[f'week_{w:02d}_rh_high_days'].mean(),
        'rad_mean'        : df_fitur[f'week_{w:02d}_rad_mean'].mean(),
    }
    weekly_summary.append(row)
df_weekly_summary = pd.DataFrame(weekly_summary)

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
fig.suptitle('Profil Agroklimatologi Mingguan Siklus Tanam 16 Minggu', fontsize=13, fontweight='bold')

axes[0, 0].plot(df_weekly_summary['week_no'], df_weekly_summary['rain_total_mean'], marker='o', color='#2980b9')
axes[0, 0].set_title('Total Curah Hujan Mingguan')
axes[0, 0].set_ylabel('mm/minggu')

axes[0, 1].plot(df_weekly_summary['week_no'], df_weekly_summary['dry_spell_mean'], marker='o', color='#d35400')
axes[0, 1].set_title('Deret Hari Kering Maksimum')
axes[0, 1].set_ylabel('hari')

axes[1, 0].plot(df_weekly_summary['week_no'], df_weekly_summary['heat_days_mean'], marker='o', color='#c0392b', label='Heat days')
axes[1, 0].plot(df_weekly_summary['week_no'], df_weekly_summary['rh_high_mean'], marker='s', color='#16a085', label='RH tinggi')
axes[1, 0].set_title('Hari Stres Panas dan Kelembapan Tinggi')
axes[1, 0].set_ylabel('hari/minggu')
axes[1, 0].legend()

axes[1, 1].plot(df_weekly_summary['week_no'], df_weekly_summary['rad_mean'], marker='o', color='#f39c12')
axes[1, 1].axhline(LOW_RAD_MJ, color='#7f8c8d', linestyle='--', linewidth=1, label='Ambang rendah')
axes[1, 1].set_title('Radiasi Matahari Rata-rata')
axes[1, 1].set_ylabel('MJ/m²/hari')
axes[1, 1].legend()

for ax in axes.flat:
    ax.set_xticks(range(1, N_WEEKS_CYCLE + 1))
    ax.set_xlabel('Minggu setelah tanam')
    for fase, weeks in FASE_MINGGU.items():
        ax.axvspan(min(weeks)-0.5, max(weeks)+0.5, alpha=0.05)

plt.tight_layout()
save_figure('profil_agroklimatologi_mingguan.png')
plt.show()


### 3.5 Korelasi Fitur Agroklimatologi dan Fase dengan Produktivitas

Korelasi dihitung untuk fitur agroklimatologi terpilih dan fitur musiman lama. Analisis ini bersifat eksploratif, bukan bukti kausal. Namun arah korelasi membantu menilai apakah indikator biologis seperti dry spell, heat stress, disease risk, dan radiasi rendah membawa sinyal yang lebih tajam daripada agregasi rata-rata musiman.


In [ ]:
corr_cols = KOLOM_CUACA_AGRO_MODEL + KOLOM_CUACA_LAMA + ['luas_panen_ha', 'produktivitas_ton_per_ha']
korel = df_fitur[corr_cols].corr(numeric_only=True)
korel_target = korel['produktivitas_ton_per_ha'].drop('produktivitas_ton_per_ha').sort_values()

fig, axes = plt.subplots(1, 2, figsize=(15, 7))

bottom = korel_target.head(15)
top    = korel_target.tail(15)
plot_corr = pd.concat([bottom, top])
colors = ['#c0392b' if v < 0 else '#27ae60' for v in plot_corr.values]
axes[0].barh(range(len(plot_corr)), plot_corr.values, color=colors, alpha=0.85)
axes[0].set_yticks(range(len(plot_corr)))
axes[0].set_yticklabels([c.replace('week_', 'w').replace('phase_', 'ph_') for c in plot_corr.index], fontsize=8)
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title('Korelasi Terkuat dengan Produktivitas')
axes[0].set_xlabel('Pearson r')

phase_focus = [c for c in KOLOM_PHASE_MODEL + KOLOM_STRESS_MODEL if c in korel_target.index]
phase_corr = korel_target.loc[phase_focus].sort_values().tail(20)
colors_phase = ['#c0392b' if v < 0 else '#27ae60' for v in phase_corr.values]
axes[1].barh(range(len(phase_corr)), phase_corr.values, color=colors_phase, alpha=0.85)
axes[1].set_yticks(range(len(phase_corr)))
axes[1].set_yticklabels([c.replace('phase_', '').replace('stress_', 'stress:') for c in phase_corr.index], fontsize=8)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Fitur Fase/Stres Paling Informatif')
axes[1].set_xlabel('Pearson r')

plt.tight_layout()
save_figure('korelasi_agroklimatologi_target.png')
plt.show()

print("Top korelasi positif:")
print(korel_target.tail(10).round(3).to_string())
print()
print("Top korelasi negatif:")
print(korel_target.head(10).round(3).to_string())


---
## Section 4 – Pemodelan dan Evaluasi

### 4.1 Konfigurasi Model

#### Model ML

| Model | Alasan Pemilihan |
|---|---|
| **Ridge Regression** | Baseline linear; regularisasi L2 efektif pada rasio sampel/fitur rendah |
| **Random Forest** | Ensemble pohon; robust terhadap sampel kecil, menghasilkan feature importance |
| **Gradient Boosting** | Umumnya akurasi terbaik pada data tabular; diregularisasi `max_depth` & `subsample` |

#### Model Baseline (Acuan Minimum)

Setiap model ML *harus* melampaui baseline ini agar dianggap berguna:

| Baseline | Deskripsi |
|---|---|
| **Naive Mean** | Prediksi = rata-rata produktivitas seluruh data training |
| **Naive Kab Mean** | Prediksi = rata-rata historis produktivitas per kabupaten |

> **Mengapa baseline penting?** Target produktivitas memiliki koefisien variasi
> hanya ~9%. Artinya prediksi naif "selalu rata-rata" sudah menghasilkan MAPE ≈ 9%
> tanpa belajar apapun. Model ML baru bermakna jika MAPE-nya jauh lebih kecil dari ini.

**Catatan model sequence:** fitur sudah dibuat sequence-aware melalui kolom minggu 1–16, tetapi notebook tidak memaksakan LSTM/deep learning karena ukuran data hanya sekitar 90 sampel. Untuk ukuran ini, Ridge dan model tree-based yang dievaluasi dengan walk-forward validation lebih realistis dan lebih mudah diaudit secara ilmiah. Eksperimen deep learning sebaiknya baru dipertimbangkan bila tersedia deret multi-tahun yang jauh lebih panjang atau unit observasi yang lebih banyak.


In [ ]:
# ── Model ML ─────────────────────────────────────────────────────────────────
MODELDEFS = {
    'Ridge Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model',  Ridge(alpha=10.0)),
    ]),
    'Ridge Hist Lag': Pipeline([
        ('scaler', StandardScaler()),
        ('model',  Ridge(alpha=3.0)),
    ]),
    'Random Forest': RandomForestRegressor(
        n_estimators     = 300,
        max_depth        = 5,
        min_samples_leaf = 3,
        max_features     = 'sqrt',
        random_state     = RANDOM_STATE,
        n_jobs           = -1,
    ),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators     = 150,
        max_depth        = 3,
        learning_rate    = 0.08,
        subsample        = 0.8,
        min_samples_leaf = 3,
        random_state     = RANDOM_STATE,
    ),
}

# ── Model Baseline ────────────────────────────────────────────────────────────
class NaiveMeanRegressor:
    """Prediksi selalu rata-rata target dari data training."""
    def fit(self, X, y, **kw):
        self.mean_ = float(np.mean(y))
        return self
    def predict(self, X):
        return np.full(len(X), self.mean_)

class NaiveKabMeanRegressor:
    """Prediksi rata-rata historis per kabupaten dari data training."""
    def fit(self, X_meta, y, **kw):
        # X_meta: DataFrame dengan kolom 'kabupaten' dan 'produktivitas_ton_per_ha'
        self.kab_mean_  = X_meta.groupby('kabupaten')['produktivitas_ton_per_ha'].mean()
        self.global_mean_ = float(np.mean(y))
        return self
    def predict(self, X_meta):
        return (X_meta['kabupaten']
                .map(self.kab_mean_)
                .fillna(self.global_mean_)
                .values)

class NaiveLag1Regressor:
    """Baseline temporal: prediksi produktivitas = produktivitas tahun sebelumnya."""
    def fit(self, X_meta, y=None, **kw):
        return self
    def predict(self, X_meta):
        return X_meta['prodvt_lag1'].values

class NaiveRoll2Regressor:
    """Baseline temporal: prediksi = rata-rata produktivitas dua tahun terakhir."""
    def fit(self, X_meta, y=None, **kw):
        return self
    def predict(self, X_meta):
        return X_meta['prodvt_roll2'].values

MODEL_FEATURES = {name: FITUR_MODEL for name in MODELDEFS}
MODEL_FEATURES['Ridge Hist Lag'] = FITUR_HIST

BASELINEDEFS = {
    'Naive Mean'    : NaiveMeanRegressor(),
    'Naive Kab Mean': NaiveKabMeanRegressor(),
    'Naive Lag1'    : NaiveLag1Regressor(),
    'Naive Roll2'   : NaiveRoll2Regressor(),
}

SEMUA_MODEL = list(MODELDEFS.keys()) + list(BASELINEDEFS.keys())

print("Konfigurasi model ML:")
for name, mdl in MODELDEFS.items():
    if hasattr(mdl, 'steps'):
        params = mdl.named_steps['model'].get_params()
    else:
        params = {k: v for k, v in mdl.get_params().items()
                  if k in ('n_estimators', 'max_depth', 'learning_rate',
                            'subsample', 'alpha', 'min_samples_leaf')}
    print(f"  {name}: {params}")
print()
print("Baseline model: Naive Mean, Naive Kab Mean, Naive Lag1, Naive Roll2")


### 4.2 Walk-Forward Validation

**[FIX v3] Penggantian LOYOCV → Walk-Forward Validation**

Masalah pada LOYOCV versi sebelumnya: fold pertama menguji tahun 2019 tetapi
dilatih menggunakan tahun 2020–2024 (data *masa depan*) — ini adalah **temporal leakage**.

Walk-Forward Validation menjamin data training selalu berasal dari tahun *sebelum*
tahun test:

```
Fold 1: Train [2019]           → Test 2020  (15 train, 15 test)
Fold 2: Train [2019–2020]      → Test 2021  (30 train, 15 test)
Fold 3: Train [2019–2021]      → Test 2022  (45 train, 15 test)
Fold 4: Train [2019–2022]      → Test 2023  (60 train, 15 test)
Fold 5: Train [2019–2023]      → Test 2024  (75 train, 15 test)
```

> **Catatan fold awal:** Fold 1 hanya memiliki 15 sampel training untuk
> fitur agroklimatologi yang relatif banyak — rasio sangat rendah. Ridge (regularisasi kuat) lebih stabil
> di kondisi ini; RF dan GB di fold awal perlu diinterpretasikan dengan hati-hati.

In [ ]:
def hitung_mape(y_true, y_pred, eps=1e-8):
    return np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + eps))) * 100

# Struktur penyimpanan hasil
WFV_HASIL = {
    nm: {'rmse': [], 'mae': [], 'r2': [], 'mape': [],
         'tahun': [], 'y_true': [], 'y_pred': []}
    for nm in SEMUA_MODEL
}

TAHUN_TEST_WFV = TAHUN_FITUR[1:]   # test mulai 2020 (butuh ≥1 tahun training)

print("Walk-Forward Validation")
print("=" * 65)

for tahun_test in TAHUN_TEST_WFV:
    train_mask = df_fitur['tahun'] < tahun_test
    test_mask  = df_fitur['tahun'] == tahun_test

    y_tr = df_model[train_mask]['produktivitas_ton_per_ha'].values
    y_te = df_model[test_mask]['produktivitas_ton_per_ha'].values

    meta_tr = df_fitur[train_mask]
    meta_te = df_fitur[test_mask]

    tahun_train_str = (f"{TAHUN_FITUR[0]}–{tahun_test-1}"
                       if tahun_test - 1 > TAHUN_FITUR[0]
                       else str(TAHUN_FITUR[0]))
    print(f"  Fold {tahun_test} | Train: {tahun_train_str} "
          f"({len(y_tr):>2} sampel) | Test: {tahun_test} (15 sampel)")

    # ── Model ML ──────────────────────────────────────────────────────────────
    for name, mdl_def in MODELDEFS.items():
        fitur_cols = MODEL_FEATURES[name]
        X_tr = df_model.loc[train_mask, fitur_cols].values.astype(float)
        X_te = df_model.loc[test_mask, fitur_cols].values.astype(float)

        m    = copy.deepcopy(mdl_def)
        m.fit(X_tr, y_tr)
        y_pr = np.clip(m.predict(X_te), 0, None)

        WFV_HASIL[name]['rmse'].append(np.sqrt(mean_squared_error(y_te, y_pr)))
        WFV_HASIL[name]['mae'].append(mean_absolute_error(y_te, y_pr))
        WFV_HASIL[name]['r2'].append(r2_score(y_te, y_pr))
        WFV_HASIL[name]['mape'].append(hitung_mape(y_te, y_pr))
        WFV_HASIL[name]['tahun'].append(tahun_test)
        WFV_HASIL[name]['y_true'].extend(y_te.tolist())
        WFV_HASIL[name]['y_pred'].extend(y_pr.tolist())

    # ── Baseline ──────────────────────────────────────────────────────────────
    for name, baseline_def in BASELINEDEFS.items():
        b = copy.deepcopy(baseline_def)
        b.fit(meta_tr, y_tr)
        y_pr_base = b.predict(meta_te)
        WFV_HASIL[name]['rmse'].append(np.sqrt(mean_squared_error(y_te, y_pr_base)))
        WFV_HASIL[name]['mae'].append(mean_absolute_error(y_te, y_pr_base))
        WFV_HASIL[name]['r2'].append(r2_score(y_te, y_pr_base))
        WFV_HASIL[name]['mape'].append(hitung_mape(y_te, y_pr_base))
        WFV_HASIL[name]['tahun'].append(tahun_test)
        WFV_HASIL[name]['y_true'].extend(y_te.tolist())
        WFV_HASIL[name]['y_pred'].extend(y_pr_base.tolist())

print()
print("Selesai.")


### 4.3 Perbandingan Performa Model (Walk-Forward CV)

In [ ]:
# Tabel ringkasan
rows_cv = []
for name in SEMUA_MODEL:
    h = WFV_HASIL[name]
    rows_cv.append({
        'Model'         : name,
        'RMSE mean'     : np.mean(h['rmse']),
        'RMSE std'      : np.std(h['rmse']),
        'MAE mean'      : np.mean(h['mae']),
        'R2 mean'       : np.mean(h['r2']),
        'MAPE mean (%)' : np.mean(h['mape']),
        'MAPE std (%)'  : np.std(h['mape']),
    })

df_cv_summary = (pd.DataFrame(rows_cv)
                 .set_index('Model')
                 .sort_values('MAPE mean (%)'))

print("Hasil Walk-Forward Validation (Produktivitas ton/ha)  –  5 fold (2020–2024)")
print("=" * 72)
print(df_cv_summary.round(4).to_string())
print()

# Baseline MAPE sebagai referensi
mape_naive_mean = df_cv_summary.loc['Naive Mean', 'MAPE mean (%)']
mape_naive_kab  = df_cv_summary.loc['Naive Kab Mean', 'MAPE mean (%)']
baseline_models = list(BASELINEDEFS.keys())
BEST_BASELINE_WFV = df_cv_summary.loc[baseline_models, 'MAPE mean (%)'].idxmin()
mape_best_baseline = df_cv_summary.loc[BEST_BASELINE_WFV, 'MAPE mean (%)']
print(f"Referensi Baseline:")
print(f"  Naive Mean      : MAPE = {mape_naive_mean:.2f}%  ← batas bawah 'tidak ada informasi'")
print(f"  Naive Kab Mean  : MAPE = {mape_naive_kab:.2f}%  ← baseline lokasi historis")
print(f"  Best Baseline   : {BEST_BASELINE_WFV} (MAPE = {mape_best_baseline:.2f}%)")
print()

# Pisahkan pemenang ML dari pemenang keseluruhan.
# Ini mencegah klaim metodologis yang keliru ketika baseline temporal lebih baik dari model ML.
ml_models_only = df_cv_summary.loc[list(MODELDEFS.keys())]
MODEL_TERBAIK  = ml_models_only['MAPE mean (%)'].idxmin()  # alias lama: model ML terbaik
MODEL_ML_TERBAIK = MODEL_TERBAIK
MODEL_OVERALL_TERBAIK = df_cv_summary['MAPE mean (%)'].idxmin()
mape_terbaik   = ml_models_only.loc[MODEL_ML_TERBAIK, 'MAPE mean (%)']
mape_overall   = df_cv_summary.loc[MODEL_OVERALL_TERBAIK, 'MAPE mean (%)']

print(f"Model ML Terbaik        : {MODEL_ML_TERBAIK}  (MAPE = {mape_terbaik:.2f}%)")
print(f"Model Overall Terbaik   : {MODEL_OVERALL_TERBAIK}  (MAPE = {mape_overall:.2f}%)")

if mape_terbaik < mape_naive_kab:
    selisih_kab = mape_naive_kab - mape_terbaik
    print(f"  → Model ML unggul {selisih_kab:.2f}% poin di atas Naive Kab Mean")
else:
    selisih_kab = mape_terbaik - mape_naive_kab
    print(f"  [CATATAN] Model ML masih kalah {selisih_kab:.2f}% poin dari Naive Kab Mean.")

if mape_terbaik < mape_best_baseline:
    selisih_base = mape_best_baseline - mape_terbaik
    print(f"  → Model ML juga unggul {selisih_base:.2f}% poin di atas baseline terbaik")
else:
    selisih_base = mape_terbaik - mape_best_baseline
    print(f"  [BATASAN] Model ML belum mengalahkan baseline temporal terbaik ({BEST_BASELINE_WFV}); "
          f"selisih {selisih_base:.2f}% poin.")


In [ ]:
# Visualisasi perbandingan per fold
model_colors = {
    'Ridge Regression'  : '#3498db',
    'Ridge Hist Lag'    : '#1abc9c',
    'Random Forest'     : '#2ecc71',
    'Gradient Boosting' : '#e74c3c',
    'Naive Mean'        : '#95a5a6',
    'Naive Kab Mean'    : '#bdc3c7',
    'Naive Lag1'        : '#7f8c8d',
    'Naive Roll2'       : '#34495e',
}
metrik_plot = [
    ('mape', 'MAPE (%)',      'lower is better'),
    ('rmse', 'RMSE (ton/ha)', 'lower is better'),
    ('mae',  'MAE (ton/ha)',  'lower is better'),
    ('r2',   'R²',            'higher is better'),
]

n_fold = len(TAHUN_TEST_WFV)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Perbandingan Model – Walk-Forward Validation (2020–2024)\n'
             '(Target: Produktivitas ton/ha  |  Abu-abu = Baseline)',
             fontsize=12, fontweight='bold')

for ax, (metrik, ylabel, note) in zip(axes.flat, metrik_plot):
    x_pos = np.arange(n_fold)
    n_mdl = len(SEMUA_MODEL)
    width = 0.75 / n_mdl

    for i, name in enumerate(SEMUA_MODEL):
        vals    = WFV_HASIL[name][metrik]
        is_base = name.startswith('Naive')
        alpha   = 0.5 if is_base else 0.85
        hatch   = '//' if is_base else ''
        ax.bar(x_pos + i * width, vals, width=width,
               color=model_colors.get(name, '#7f8c8d'), alpha=alpha,
               label=name, hatch=hatch)

    ax.set_xticks(x_pos + width * (n_mdl - 1) / 2)
    ax.set_xticklabels([str(t) for t in TAHUN_TEST_WFV], fontsize=9)
    ax.set_xlabel('Tahun Test (fold)')
    ax.set_ylabel(ylabel)
    ax.set_title(f'{ylabel}  –  {note}', fontsize=9)

axes.flat[0].legend(fontsize=7, loc='upper right')
plt.tight_layout()
save_figure('perbandingan_model_wfv.png')
plt.show()


### 4.3b Ablation Study – Musiman Lama vs Agroklimatologi Baru

Ablation ini menjadi bagian kunci revisi. Tujuannya membandingkan apakah fitur biologis berbasis harian-mingguan memberi informasi tambahan dibanding agregasi musiman lama, sambil tetap menjaga baseline historis dan efek kabupaten.

Kelompok yang diuji:

- histori produktivitas saja;
- fitur musiman lama;
- fitur agroklimatologi baru;
- indikator stres spesifik seperti heat stress, dry spell, kelembapan penyakit, dan radiasi rendah;
- kombinasi dengan kabupaten untuk melihat apakah sinyal cuaca tetap berguna setelah efek lokasi dimasukkan.


In [ ]:
KOLOM_HEAT_STRESS = [c for c in KOLOM_CUACA_AGRO_MODEL if c.endswith('_heat_days') or 'generatif_heat' in c]
KOLOM_DRY_SPELL   = [c for c in KOLOM_CUACA_AGRO_MODEL if c.endswith('_dry_spell_max') or 'hot_and_dry' in c]
KOLOM_DISEASE_RH  = [c for c in KOLOM_CUACA_AGRO_MODEL if c.endswith('_rh_high_days') or c.endswith('_disease_risk') or 'disease_risk' in c]
KOLOM_RADIATION   = [c for c in KOLOM_CUACA_AGRO_MODEL if c.endswith('_rad_mean') or c.endswith('_rad_low_days')]
KOLOM_RAIN_AGRO   = [c for c in KOLOM_CUACA_AGRO_MODEL if ('rain_' in c or c.endswith('_dry_spell_max'))]

FEATURE_SETS = {
    'Histori target ringkas'      : FITUR_HIST,
    'Histori + Kabupaten'        : FITUR_HIST_KAB,
    'Musiman lama saja'           : FITUR_OLD_WEATHER,
    'Histori + Musiman lama'      : FITUR_OLD_HIST,
    'Histori + Musiman lama + Kab': FITUR_OLD_HIST_KAB,
    'Agro baru saja'              : FITUR_AGRO_ONLY,
    'Histori + Agro baru'         : FITUR_AGRO_HIST,
    'Histori + Agro baru + Kab'   : FITUR_AGRO_HIST_KAB,
    'Agro + Luas + Kab'           : FITUR_AGRO_FULL,
    'Heat stress agro'            : FITUR_HIST + KOLOM_HEAT_STRESS,
    'Dry spell agro'              : FITUR_HIST + KOLOM_DRY_SPELL,
    'Humidity disease agro'       : FITUR_HIST + KOLOM_DISEASE_RH,
    'Radiation agro'              : FITUR_HIST + KOLOM_RADIATION,
    'Rain variability agro'       : FITUR_HIST + KOLOM_RAIN_AGRO,
}


def evaluate_wfv_estimator(estimator, feature_cols):
    rows = []
    for tahun_test in TAHUN_TEST_WFV:
        train_mask = df_fitur['tahun'] < tahun_test
        test_mask  = df_fitur['tahun'] == tahun_test
        X_tr = df_model.loc[train_mask, feature_cols].values.astype(float)
        y_tr = df_model.loc[train_mask, 'produktivitas_ton_per_ha'].values
        X_te = df_model.loc[test_mask, feature_cols].values.astype(float)
        y_te = df_model.loc[test_mask, 'produktivitas_ton_per_ha'].values

        m = copy.deepcopy(estimator)
        m.fit(X_tr, y_tr)
        y_pr = np.clip(m.predict(X_te), 0, None)
        rows.append({
            'tahun_test': tahun_test,
            'RMSE': np.sqrt(mean_squared_error(y_te, y_pr)),
            'MAE' : mean_absolute_error(y_te, y_pr),
            'R2'  : r2_score(y_te, y_pr),
            'MAPE': hitung_mape(y_te, y_pr),
        })
    return pd.DataFrame(rows)

ablation_rows = []
for feature_set_name, cols in FEATURE_SETS.items():
    hasil = evaluate_wfv_estimator(MODELDEFS['Ridge Regression'], cols)
    ablation_rows.append({
        'Feature Set': feature_set_name,
        'Jumlah Fitur': len(cols),
        'MAPE mean (%)': hasil['MAPE'].mean(),
        'MAPE std (%)' : hasil['MAPE'].std(),
        'RMSE mean'    : hasil['RMSE'].mean(),
        'R2 mean'      : hasil['R2'].mean(),
    })

df_ablation = (pd.DataFrame(ablation_rows)
               .sort_values('MAPE mean (%)')
               .reset_index(drop=True))
print("Ablation Study Ridge Regression (Walk-Forward 2020–2024)")
print(df_ablation.round(4).to_string(index=False))
print()

mape_old = df_ablation.loc[df_ablation['Feature Set'].eq('Histori + Musiman lama + Kab'), 'MAPE mean (%)'].iloc[0]
mape_new = df_ablation.loc[df_ablation['Feature Set'].eq('Histori + Agro baru + Kab'), 'MAPE mean (%)'].iloc[0]
print("Interpretasi ablation:")
if mape_new < mape_old:
    print(f"  Fitur agroklimatologi baru lebih informatif daripada musiman lama pada konfigurasi histori+kabupaten (selisih {mape_old-mape_new:.2f} poin MAPE).")
else:
    print(f"  Fitur agroklimatologi baru belum mengungguli musiman lama pada konfigurasi histori+kabupaten (selisih {mape_new-mape_old:.2f} poin MAPE).")
print("  Baca hasil bersama jumlah fitur: performa yang sedikit lebih baik tetapi memakai jauh lebih banyak fitur belum tentu lebih stabil.")
print("  Kelompok heat/dry/RH/radiasi membantu mengidentifikasi mekanisme agroklimatologi, bukan sekadar mengejar skor.")


### 4.4 Evaluasi Final – Model Terbaik (Train 2019–2023 / Test 2024)

Evaluasi akhir menggunakan split kronologis paling realistis:  
melatih pada seluruh data historis (2019–2023) dan menguji pada tahun terbaru (2024).

> **Catatan interpretasi R²:**  
> Dua metrik R² yang dilaporkan di bawah ini terlihat kontradiktif — R²
> *produktivitas* bisa negatif sementara R² *produksi* bisa sangat tinggi.
> Ini **bukan** inkonsistensi; ini artefak matematis:  
> Produksi = Produktivitas × Luas_Panen. Variansi luas panen antar kabupaten
> jauh lebih besar daripada variansi produktivitas, sehingga R² produksi
> didominasi oleh variansi luas — bukan kemampuan prediksi model.  
> **Gunakan R² produktivitas sebagai ukuran kemampuan model yang sesungguhnya.**

In [ ]:
mask_train = df_fitur['tahun'] <= 2023
mask_test  = df_fitur['tahun'] == 2024

fitur_final = MODEL_FEATURES[MODEL_TERBAIK]
X_train  = df_model.loc[mask_train, fitur_final].values.astype(float)
y_train  = df_model.loc[mask_train, 'produktivitas_ton_per_ha'].values
X_test   = df_model.loc[mask_test, fitur_final].values.astype(float)
y_test   = df_model.loc[mask_test, 'produktivitas_ton_per_ha'].values
meta_test = df_fitur[mask_test].copy().reset_index(drop=True)

# Train model ML terbaik berdasarkan WFV
model_final = copy.deepcopy(MODELDEFS[MODEL_TERBAIK])
model_final.fit(X_train, y_train)
y_pred_prodvt = np.clip(model_final.predict(X_test), 0, None)

# Train seluruh baseline final dan pilih baseline terbaik di test 2024 untuk analisis pembanding
meta_train = df_fitur[mask_train]
baseline_final_results = []
baseline_final_preds = {}


def metrik_set(y_true, y_pred, label):
    return {
        'label': label,
        'RMSE' : np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE'  : mean_absolute_error(y_true, y_pred),
        'R2'   : r2_score(y_true, y_pred),
        'MAPE' : hitung_mape(y_true, y_pred),
    }

for name, baseline_def in BASELINEDEFS.items():
    b = copy.deepcopy(baseline_def)
    b.fit(meta_train, y_train)
    y_pred_base = b.predict(meta_test)
    baseline_final_preds[name] = y_pred_base
    baseline_final_results.append(metrik_set(y_test, y_pred_base, name))

df_final_baseline = pd.DataFrame(baseline_final_results).sort_values('MAPE').reset_index(drop=True)
BASELINE_FINAL_TERBAIK = df_final_baseline.loc[0, 'label']
y_pred_baseline_best = baseline_final_preds[BASELINE_FINAL_TERBAIK]

# Prediksi produksi (turunan) dan error analysis
meta_test['prodvt_aktual']             = y_test
meta_test['prodvt_prediksi']           = y_pred_prodvt
meta_test['prodvt_prediksi_baseline']  = y_pred_baseline_best
meta_test['residual_prodvt']           = y_pred_prodvt - y_test
meta_test['abs_error_prodvt']          = np.abs(meta_test['residual_prodvt'])
meta_test['produksi_prediksi_ton']     = y_pred_prodvt * meta_test['luas_panen_ha']
meta_test['produksi_error_ton']        = meta_test['produksi_prediksi_ton'] - meta_test['produksi_ton']
meta_test['produksi_error_pct']        = meta_test['produksi_error_ton'] / meta_test['produksi_ton'].abs().clip(lower=1e-8) * 100

# Alias backward-compatible untuk cell visualisasi lama
y_pred_nk = y_pred_baseline_best

# Metrik produktivitas
m_prodvt_ml = metrik_set(y_test, y_pred_prodvt, MODEL_TERBAIK)
m_prodvt_nk = metrik_set(y_test, y_pred_baseline_best, BASELINE_FINAL_TERBAIK)

# Metrik produksi (turunan)
y_prod_akt  = meta_test['produksi_ton'].values
y_prod_pred = meta_test['produksi_prediksi_ton'].values
rmse_prod = np.sqrt(mean_squared_error(y_prod_akt, y_prod_pred))
r2_prod   = r2_score(y_prod_akt, y_prod_pred)
mape_prod = hitung_mape(y_prod_akt, y_prod_pred)

sep = '=' * 65
print(sep)
print(f"EVALUASI FINAL – {MODEL_TERBAIK}")
print(f"Training: 2019–2023 ({X_train.shape[0]} sampel)  |  Test: 2024 (15 sampel)")
print(f"Fitur model final: {len(fitur_final)} kolom → {fitur_final}")
print(sep)
print()
print("Metrik Produktivitas (ton/ha)  ← ukuran kemampuan model sesungguhnya:")
print(f"  {'Model':<22} {'RMSE':>8} {'MAE':>8} {'R²':>8} {'MAPE':>8}")
print(f"  {'-'*56}")
for m in [m_prodvt_ml] + baseline_final_results:
    print(f"  {m['label']:<22} {m['RMSE']:>8.4f} {m['MAE']:>8.4f} "
          f"{m['R2']:>8.4f} {m['MAPE']:>7.2f}%")
print()
print(f"Baseline final terbaik: {BASELINE_FINAL_TERBAIK}")
print()
print("Metrik Produksi Turunan (ton)  ← HATI-HATI: R² dipengaruhi variansi luas panen:")
print(f"  RMSE  : {rmse_prod:,.0f} ton")
print(f"  R²    : {r2_prod:.4f}  ← tinggi karena dominasi variansi luas, BUKAN bukti model bagus")
print(f"  MAPE  : {mape_prod:.2f}%")
print()
delta_mape_final = m_prodvt_nk['MAPE'] - m_prodvt_ml['MAPE']
if delta_mape_final > 0:
    print(f"[OK] Model ML mengungguli baseline final terbaik ({BASELINE_FINAL_TERBAIK}) sebesar "
          f"{delta_mape_final:.2f} poin MAPE pada test 2024.")
else:
    print(f"[PERINGATAN METODOLOGIS] Model ML belum mengungguli baseline final terbaik ({BASELINE_FINAL_TERBAIK}) "
          f"pada test 2024 (selisih {abs(delta_mape_final):.2f} poin MAPE).")


---
## Section 5 – Visualisasi Final

### 5.1 Scatter Plot Prediksi vs Aktual (Test 2024)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'{MODEL_TERBAIK} – Data Test 2024\n'
             f'R² = {m_prodvt_ml["R2"]:.4f}  |  MAPE = {m_prodvt_ml["MAPE"]:.2f}%',
             fontsize=11, fontweight='bold')

# Scatter produktivitas
ax = axes[0]
ax.scatter(y_test, y_pred_prodvt, color='steelblue', alpha=0.75,
           s=65, edgecolors='white', linewidth=0.5, zorder=3, label=MODEL_TERBAIK)
ax.scatter(y_test, y_pred_nk, color='#95a5a6', alpha=0.6,
           s=45, marker='^', zorder=2, label=BASELINE_FINAL_TERBAIK)
lim = [min(y_test.min(), y_pred_prodvt.min()) * 0.95,
       max(y_test.max(), y_pred_prodvt.max()) * 1.05]
ax.plot(lim, lim, '--', color='crimson', linewidth=1.5, label='Ideal')
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel('Aktual (ton/ha)')
ax.set_ylabel('Prediksi (ton/ha)')
ax.set_title(f'Produktivitas  |  RMSE = {m_prodvt_ml["RMSE"]:.3f} ton/ha')
ax.legend(fontsize=8)

# Scatter produksi turunan
ax = axes[1]
ax.scatter(y_prod_akt / 1e3, y_prod_pred / 1e3,
           color='darkorange', alpha=0.75, s=65, edgecolors='white', linewidth=0.5)
lim2 = [min(y_prod_akt.min(), y_prod_pred.min()) / 1e3 * 0.9,
        max(y_prod_akt.max(), y_prod_pred.max()) / 1e3 * 1.05]
ax.plot(lim2, lim2, '--', color='crimson', linewidth=1.5)
ax.set_xlim(lim2); ax.set_ylim(lim2)
ax.set_xlabel('Aktual (ribu ton)')
ax.set_ylabel('Prediksi (ribu ton)')
ax.set_title(f'Produksi Turunan  |  RMSE = {rmse_prod/1e3:.1f} ribu ton  |  MAPE = {mape_prod:.2f}%\n'
             f'(R²={r2_prod:.3f} — tinggi karena dominasi variansi luas panen)')
ax.tick_params(labelsize=8)

plt.tight_layout()
save_figure('scatter_prediksi_aktual.png')
plt.show()


### 5.1b Residual dan Signed Error Analysis

Visualisasi ini ditambahkan agar evaluasi tidak hanya bergantung pada MAPE; signed error menunjukkan arah bias over/under-prediction per kabupaten.


In [ ]:
residual_plot = meta_test.sort_values('residual_prodvt')
colors_res = ['#e74c3c' if v > 0 else '#3498db' for v in residual_plot['residual_prodvt']]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax = axes[0]
ax.barh(residual_plot['kabupaten'], residual_plot['residual_prodvt'], color=colors_res, alpha=0.85)
ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('Residual Produktivitas (prediksi - aktual, ton/ha)')
ax.set_title('Signed Residual Produktivitas – Test 2024')
ax.tick_params(axis='y', labelsize=8)

ax = axes[1]
err_pct_plot = meta_test.sort_values('produksi_error_pct')
colors_pct = ['#e74c3c' if v > 0 else '#3498db' for v in err_pct_plot['produksi_error_pct']]
ax.barh(err_pct_plot['kabupaten'], err_pct_plot['produksi_error_pct'], color=colors_pct, alpha=0.85)
ax.axvline(0, color='black', linewidth=1)
ax.axvline(15, color='#f39c12', linestyle='--', linewidth=1, alpha=0.7)
ax.axvline(-15, color='#f39c12', linestyle='--', linewidth=1, alpha=0.7)
ax.set_xlabel('Signed Error Produksi (%)')
ax.set_title('Arah Bias Produksi Turunan – Test 2024')
ax.tick_params(axis='y', labelsize=8)

fig.suptitle('Residual Analysis: Biru = under-prediction, Merah = over-prediction',
             fontsize=11, fontweight='bold')
plt.tight_layout()
save_figure('residual_analysis_2024.png')
plt.show()


### 5.2 Perbandingan Produksi Aktual vs Prediksi per Kabupaten (2024)

In [ ]:
kab_sorted = meta_test.sort_values('produksi_ton', ascending=False)['kabupaten'].values
x_pos      = np.arange(len(kab_sorted))
width      = 0.38

akt_vals  = [meta_test[meta_test['kabupaten'] == k]['produksi_ton'].values[0] / 1e3
             for k in kab_sorted]
pred_vals = [meta_test[meta_test['kabupaten'] == k]['produksi_prediksi_ton'].values[0] / 1e3
             for k in kab_sorted]
mape_vals = [abs(a - p) / max(a, 1e-8) * 100 for a, p in zip(akt_vals, pred_vals)]

fig, ax = plt.subplots(figsize=(16, 6))
ax.bar(x_pos - width / 2, akt_vals,  width=width, color='steelblue',
       alpha=0.85, label='Aktual')
ax.bar(x_pos + width / 2, pred_vals, width=width, color='darkorange',
       alpha=0.85, label='Prediksi')

for i, (a, p, mp) in enumerate(zip(akt_vals, pred_vals, mape_vals)):
    clr = '#27ae60' if mp < 15 else ('#f39c12' if mp < 30 else '#e74c3c')
    ax.text(x_pos[i], max(a, p) + 1.5, f'{mp:.1f}%',
            ha='center', va='bottom', fontsize=6.5, color=clr, fontweight='bold')

ax.set_xticks(x_pos)
ax.set_xticklabels(kab_sorted, rotation=40, ha='right', fontsize=8)
ax.set_ylabel('Produksi (ribu ton)')
ax.set_title(f'Perbandingan Produksi Aktual vs Prediksi per Kabupaten – Test 2024\n'
             f'(Angka di atas bar: MAPE – Hijau < 15%  |  Oranye 15–30%  |  Merah > 30%)',
             fontsize=10)
ax.legend(fontsize=9)
plt.tight_layout()
save_figure('bar_per_kabupaten_2024.png')
plt.show()


### 5.3 Feature Importance

In [ ]:
def kategori_fitur_agro(c):
    if c.startswith('kabupaten_'):
        return 'Kabupaten (OHE)'
    if c.startswith('prodvt_'):
        return 'Histori Target'
    if c == 'luas_panen_ha':
        return 'Luas Panen'
    if c.startswith('old_'):
        return 'Cuaca Musiman Lama'
    if c.startswith('week_'):
        if 'rain' in c or 'dry_spell' in c:
            return 'Mingguan: Hujan/Kering'
        if 'heat' in c or 'tmean' in c or 'tmax' in c or 'gdd' in c:
            return 'Mingguan: Suhu/Panas'
        if 'rh_' in c or 'disease' in c:
            return 'Mingguan: Kelembapan'
        if 'rad_' in c:
            return 'Mingguan: Radiasi'
        return 'Mingguan: Lainnya'
    if c.startswith('phase_'):
        parts = c.split('_')
        if len(parts) > 2:
            return 'Fase: ' + parts[1].title()
        return 'Fase Pertumbuhan'
    if c.startswith('stress_'):
        return 'Indikator Stres Biologis'
    return 'Lainnya'

if hasattr(model_final, 'feature_importances_'):
    importances = model_final.feature_importances_
elif hasattr(model_final, 'named_steps'):
    inner = model_final.named_steps.get('model')
    importances = np.abs(inner.coef_) if hasattr(inner, 'coef_') else None
else:
    importances = None

if importances is not None:
    fitur_importance = fitur_final if 'fitur_final' in globals() else FITUR_MODEL
    df_imp = pd.DataFrame({
        'fitur'     : fitur_importance,
        'importance': importances,
    }).sort_values('importance', ascending=False)

    df_imp['kategori'] = df_imp['fitur'].apply(kategori_fitur_agro)
    df_agg_imp = df_imp.groupby('kategori')['importance'].sum().sort_values(ascending=True)

    fig, axes = plt.subplots(1, 2, figsize=(15, 7))
    fig.suptitle(f'Feature Importance – {MODEL_TERBAIK}', fontsize=12, fontweight='bold')

    top20 = df_imp.head(20)
    color_map = {
        'Histori Target': '#16a085',
        'Kabupaten (OHE)': '#95a5a6',
        'Mingguan: Hujan/Kering': '#2980b9',
        'Mingguan: Suhu/Panas': '#c0392b',
        'Mingguan: Kelembapan': '#27ae60',
        'Mingguan: Radiasi': '#f39c12',
        'Indikator Stres Biologis': '#8e44ad',
        'Cuaca Musiman Lama': '#7f8c8d',
    }
    colors_imp = [color_map.get(kategori_fitur_agro(c), '#34495e') for c in top20['fitur']]
    axes[0].barh(range(len(top20)), top20['importance'].values, color=colors_imp, alpha=0.85)
    axes[0].set_yticks(range(len(top20)))
    axes[0].set_yticklabels(
        [c.replace('kabupaten_', 'kab:')
          .replace('prodvt_', 'hist_')
          .replace('phase_', 'ph_')
          .replace('week_', 'w')
         for c in top20['fitur']],
        fontsize=8
    )
    axes[0].set_title('Top-20 Fitur Individual', fontsize=10)
    axes[0].set_xlabel('Importance')

    colors_cat = [color_map.get(c, '#34495e') for c in df_agg_imp.index]
    axes[1].barh(range(len(df_agg_imp)), df_agg_imp.values, color=colors_cat, alpha=0.85)
    axes[1].set_yticks(range(len(df_agg_imp)))
    axes[1].set_yticklabels(df_agg_imp.index, fontsize=9)
    axes[1].set_title('Importance Agregasi per Kategori', fontsize=10)
    axes[1].set_xlabel('Importance Total')
    for i, v in enumerate(df_agg_imp.values):
        axes[1].text(v + 0.001, i, f'{v:.3f}', va='center', fontsize=8)

    plt.tight_layout()
    save_figure('feature_importance.png')
    plt.show()

    print("Interpretasi singkat importance:")
    print("  Kelompok histori yang dominan berarti persistensi lokasi/tahun masih kuat.")
    print("  Fitur mingguan/fase yang muncul di top-20 memberi kandidat fase sensitif, tetapi tetap perlu dibaca sebagai asosiasi prediktif, bukan kausalitas.")
else:
    print("Model tidak menyediakan feature importances.")


### 5.4 MAPE per Kabupaten – Data Test 2024

In [ ]:
meta_test['mape_produksi'] = meta_test['produksi_error_pct'].abs()
mape_kab = meta_test[['kabupaten', 'mape_produksi']].sort_values('mape_produksi')

fig, ax = plt.subplots(figsize=(10, 7))
colors_mape = ['#2ecc71' if v < 15 else ('#f39c12' if v < 30 else '#e74c3c')
               for v in mape_kab['mape_produksi']]
bars = ax.barh(range(len(mape_kab)), mape_kab['mape_produksi'].values,
               color=colors_mape, alpha=0.88)
ax.set_yticks(range(len(mape_kab)))
ax.set_yticklabels(mape_kab['kabupaten'].values, fontsize=9)
for bar, val in zip(bars, mape_kab['mape_produksi'].values):
    ax.text(val + 0.2, bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}%', va='center', ha='left', fontsize=8.5, fontweight='bold')
ax.axvline(15, color='#27ae60', linestyle='--', linewidth=1.2, alpha=0.7, label='15%')
ax.axvline(30, color='#e67e22', linestyle='--', linewidth=1.2, alpha=0.7, label='30%')
ax.set_xlabel('MAPE (%)')
ax.set_title(f'MAPE Produksi per Kabupaten – Data Test 2024\n'
             f'Hijau < 15%  |  Oranye 15–30%  |  Merah > 30%', fontsize=10)
ax.set_xlim(0, mape_kab['mape_produksi'].max() * 1.15)
ax.legend(fontsize=8)
plt.tight_layout()
save_figure('mape_per_kabupaten.png')
plt.show()


### 5.5 Ringkasan Akhir

In [ ]:
zona_hijau  = (mape_kab['mape_produksi'] < 15).sum()
zona_oranye = ((mape_kab['mape_produksi'] >= 15) & (mape_kab['mape_produksi'] < 30)).sum()
zona_merah  = (mape_kab['mape_produksi'] >= 30).sum()


def extract_model_params(model):
    """Ambil parameter utama, termasuk estimator di dalam sklearn Pipeline."""
    interesting = {'n_estimators', 'max_depth', 'learning_rate', 'subsample',
                   'min_samples_leaf', 'alpha', 'max_features'}
    if hasattr(model, 'named_steps') and 'model' in model.named_steps:
        inner = model.named_steps['model']
        params = inner.get_params()
    elif hasattr(model, 'get_params'):
        params = model.get_params()
    else:
        return {}
    return {k: v for k, v in params.items() if k in interesting}

# Parameter model ML terbaik
params_final = extract_model_params(model_final)

sep = '=' * 65
print(sep)
print("RINGKASAN AKHIR – PREDIKSI PRODUKTIVITAS & PRODUKSI PADI")
print("Provinsi Lampung | Machine Learning Berbasis Agroklimatologi 16 Minggu")
print(sep)
print(f"  Tanggal generate  : {datetime.date.today()}")
print(f"  Cakupan data      : 15 kabupaten, 2019–2024 (90 sampel)")
print()
print("  Desain Fitur:")
print(f"    Target           : produktivitas_ton_per_ha")
print(f"    Fitur cuaca agro : {len(KOLOM_CUACA_AGRO_MODEL)} terpilih dari {len(KOLOM_CUACA_AGRO_ALL)} kandidat harian-mingguan")
print(f"    Pembanding lama  : {len(KOLOM_CUACA_LAMA)} fitur musiman agregat")
print(f"    Fitur histori    : {len(HIST_FEATURES)} kandidat lag/rolling produktivitas")
print(f"    Fitur lain       : luas_panen_ha, kabupaten (OHE × {len(KOLOM_KAB_OHE)})")
print(f"    Total fitur default: {len(FITUR_MODEL)} | fitur final ML: {len(fitur_final)}")
print()
print("  Desain Siklus Tanam:")
print(f"    Siklus representatif: 16 minggu mulai {PLANTING_MONTH_DAY[1]:02d}-{PLANTING_MONTH_DAY[0]:02d} tahun sebelumnya")
for fase, label in FASE_LABEL.items():
    print(f"    {label}")
print()
print("  Metode Evaluasi    : Walk-Forward Validation (5 fold, 2020–2024)")
print()
print("  Perbandingan Model (WFV – MAPE rata-rata):")
for name in SEMUA_MODEL:
    h = WFV_HASIL[name]
    tag = ' ← TERBAIK OVERALL' if name == MODEL_OVERALL_TERBAIK else (
          ' ← TERBAIK ML'      if name == MODEL_ML_TERBAIK else (
          ' ← BASELINE'        if name.startswith('Naive') else ''))
    print(f"    {name:<22}: MAPE {np.mean(h['mape']):.2f}% "
          f"±{np.std(h['mape']):.2f}%  R² {np.mean(h['r2']):.3f}{tag}")
print()
print(f"  Model ML Terbaik   : {MODEL_ML_TERBAIK}")
print(f"  Model Overall      : {MODEL_OVERALL_TERBAIK}")
print(f"  Parameter ML       : {params_final}")
print()
print("  Performa Final (Train 2019–2023 / Test 2024):")
print(f"    Produktivitas : RMSE={m_prodvt_ml['RMSE']:.4f} t/ha | "
      f"MAE={m_prodvt_ml['MAE']:.4f} | R²={m_prodvt_ml['R2']:.4f} | "
      f"MAPE={m_prodvt_ml['MAPE']:.2f}%")
print(f"    {BASELINE_FINAL_TERBAIK}: RMSE={m_prodvt_nk['RMSE']:.4f} t/ha | "
      f"MAE={m_prodvt_nk['MAE']:.4f} | R²={m_prodvt_nk['R2']:.4f} | "
      f"MAPE={m_prodvt_nk['MAPE']:.2f}%")
print(f"    Produksi (turunan): RMSE={rmse_prod/1e3:.1f} ribu ton | "
      f"R²={r2_prod:.4f}* | MAPE={mape_prod:.2f}%")
print(f"    * R² produksi dipengaruhi variansi luas panen, bukan akurasi model")
if delta_mape_final <= 0:
    print(f"    [BATASAN] Akurasi prediktif ML belum membaik dibanding baseline final terbaik ({BASELINE_FINAL_TERBAIK}).")
print()
print("  Ablation Study Ridge (WFV – MAPE rata-rata):")
for _, row in df_ablation.iterrows():
    print(f"    {row['Feature Set']:<24}: {row['MAPE mean (%)']:.2f}% "
          f"({int(row['Jumlah Fitur'])} fitur)")
print()
print(f"  Distribusi MAPE per Kabupaten (Test 2024):")
print(f"    Akurat   (< 15%) : {zona_hijau:>2} kabupaten")
print(f"    Moderat (15–30%) : {zona_oranye:>2} kabupaten")
print(f"    Lemah   (> 30%)  : {zona_merah:>2} kabupaten")
print(sep)
print()
print("KESIMPULAN VALIDITAS:")
print("  • Feature engineering cuaca kini lebih biologically meaningful: dry spell, heat stress, risiko penyakit berbasis RH, radiasi rendah, dan fase pertumbuhan.")
print("  • Kualitas pipeline/evaluasi tetap dijaga: input divalidasi, walk-forward validation, baseline comparison, ablation, dan residual dianalisis.")
if delta_mape_final <= 0:
    print(f"  • Namun akurasi prediktif ML BELUM membaik karena {BASELINE_FINAL_TERBAIK} masih lebih baik daripada model ML.")
else:
    print(f"  • Akurasi prediktif ML membaik terhadap baseline final terbaik ({BASELINE_FINAL_TERBAIK}) pada test final.")
